#infoPilot = multi agent

In [1]:
#install dependecies

%pip install -q -U langchain langchain-core langchain-groq requests==2.32.4 python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.0 MB/s eta 0:00:00


In [2]:
# Option A: enter keys interactively (recommended for Colab/Jupyter demos).
# getpass() hides the input so the key is never displayed or stored in the
# notebook's output.

import os
from getpass import getpass

def set_key_if_missing(var_name: str, prompt: str) -> None:
    """Prompt for an API key only if it isn't already set in the environment."""
    if not os.getenv(var_name):
        os.environ[var_name] = getpass(prompt)

set_key_if_missing("GROQ_API_KEY", "Enter GROQ API key: ")
set_key_if_missing("SERPAPI_API_KEY", "Enter SerpAPI key: ")
set_key_if_missing("MARKETAUX_API_KEY", "Enter Marketaux API key: ")
set_key_if_missing("FREEWEATHER_API_KEY", "Enter FreeWeatherAPI key: ")

Enter GROQ API key: ··········
Enter SerpAPI key: ··········
Enter Marketaux API key: ··········
Enter FreeWeatherAPI key: ··········


In [3]:
# Validate that all required keys are present WITHOUT printing their values.

REQUIRED_KEYS = [
    "GROQ_API_KEY",
    "SERPAPI_API_KEY",
    "MARKETAUX_API_KEY",
    "FREEWEATHER_API_KEY",
]

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]

if missing:
    raise EnvironmentError(
        f"Missing required API key(s): {', '.join(missing)}. "
        "Re-run the key configuration cell above and provide them."
    )

print("All required API keys are present.")
for key in REQUIRED_KEYS:
    print(f"  {key}: OK (set)")

All required API keys are present.
  GROQ_API_KEY: OK (set)
  SERPAPI_API_KEY: OK (set)
  MARKETAUX_API_KEY: OK (set)
  FREEWEATHER_API_KEY: OK (set)


In [4]:
from groq import Groq
client = Groq()

for m in client.models.list().data:
  print(m.id)

openai/gpt-oss-safeguard-20b
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-120b
whisper-large-v3
whisper-large-v3-turbo
openai/gpt-oss-20b
allam-2-7b
meta-llama/llama-prompt-guard-2-22m


In [5]:
#CONFIG MODEL NAME .SWAP THIS FOR ANY CURRENT GROQ MODEL..
from langchain_groq import ChatGroq

MODEL_NAME="openai/gpt-oss-20b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
)

print(f"LLM initialISED :{MODEL_NAME}")

LLM initialISED :openai/gpt-oss-20b


In [6]:
import requests
from langchain_core.tools import tool

WEATHER_API_URL = "https://api.weatherapi.com/v1/current.json"
WEATHER_TIMEOUT_SECONDS = 10


@tool
def get_weather(location: str) -> str:
    """
    Get current weather information for a specified location.

    Use this tool when the user asks about the weather or current weather
    conditions in a specific place (city, region, or country).

    Args:
        location: A natural-language place name, e.g. "Mumbai" or "Pune, India".
    """
    api_key = os.getenv("FREEWEATHER_API_KEY")
    if not api_key:
        return "Weather service is not configured (missing API key)."

    params = {"key": api_key, "q": location, "aqi": "no"}

    try:
        response = requests.get(WEATHER_API_URL, params=params, timeout=WEATHER_TIMEOUT_SECONDS)
    except requests.exceptions.Timeout:
        return "Weather service is currently unavailable (request timed out). Please try again later."
    except requests.exceptions.RequestException:
        return "Weather service is currently unavailable (network error). Please try again later."

    if response.status_code == 400:
        return f"Could not find weather data for '{location}'. Please check the location name."
    if response.status_code == 401 or response.status_code == 403:
        return "Weather service is currently unavailable (authentication error)."
    if response.status_code != 200:
        return f"Weather service returned an unexpected error (status {response.status_code})."

    try:
        data = response.json()
        loc = data["location"]
        current = data["current"]

        place = f"{loc.get('name', location)}, {loc.get('country', '')}".strip(", ")
        condition = current.get("condition", {}).get("text", "unknown")
        temp_c = current.get("temp_c")
        feels_like_c = current.get("feelslike_c")
        humidity = current.get("humidity")
        wind_kph = current.get("wind_kph")
        precip_mm = current.get("precip_mm")

        summary = (
            f"Current weather in {place}: {condition}, "
            f"temperature {temp_c}°C (feels like {feels_like_c}°C), "
            f"humidity {humidity}%, wind {wind_kph} kph"
        )
        if precip_mm is not None:
            summary += f", precipitation {precip_mm} mm"
        return summary + "."

    except (KeyError, ValueError, TypeError):
        return "Received an unexpected response from the weather service. Please try again later."

In [7]:
SERPAPI_URL = "https://serpapi.com/search"
SERPAPI_TIMEOUT_SECONDS = 10
SERPAPI_MAX_RESULTS = 5  # keep the context small


@tool
def web_search(query: str) -> str:
    """
    Search the web for current or general information.

    Use this for general knowledge, current events, "who is / what is"
    questions, or anything that is not specifically financial/stock-market
    news for a company (use finance_news for that instead).

    Args:
        query: The search query.
    """
    api_key = os.getenv("SERPAPI_API_KEY")
    if not api_key:
        return "Web search service is not configured (missing API key)."

    params = {"q": query, "api_key": api_key, "engine": "google"}

    try:
        response = requests.get(SERPAPI_URL, params=params, timeout=SERPAPI_TIMEOUT_SECONDS)
    except requests.exceptions.Timeout:
        return "Web search is currently unavailable (request timed out). Please try again later."
    except requests.exceptions.RequestException:
        return "Web search is currently unavailable (network error). Please try again later."

    if response.status_code != 200:
        return f"Web search service returned an unexpected error (status {response.status_code})."

    try:
        data = response.json()
        results = data.get("organic_results", [])[:SERPAPI_MAX_RESULTS]

        if not results:
            return f"No web search results found for '{query}'."

        lines = []
        for i, result in enumerate(results, start=1):
            title = result.get("title", "Untitled")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            lines.append(f"{i}. {title}\n   {snippet}\n   Source: {link}")

        return "\n".join(lines)

    except (ValueError, TypeError):
        return "Received an unexpected response from the web search service. Please try again later."

In [8]:
MARKETAUX_URL = "https://api.marketaux.com/v1/news/all"
MARKETAUX_TIMEOUT_SECONDS = 10
MARKETAUX_MAX_ARTICLES = 5  # keep the context small


@tool
def finance_news(query: str) -> str:
    """
    Search recent financial news using Marketaux.

    USE THIS TOOL whenever the user asks about financial news,
    stock news, market news, earnings, mergers, acquisitions,
    investor news, or financial developments involving a company,
    stock ticker, sector, or market.

    Examples:
    - "Latest financial news about NVIDIA"
    - "Recent news about AAPL"
    - "What is the latest news about Reliance stock?"
    - "Latest earnings news for Microsoft"

    DO NOT use web_search for financial-news queries.

    Args:
        query: The financial news search query.
    """
    api_key = os.getenv("MARKETAUX_API_KEY")
    if not api_key:
        return "Finance news service is not configured (missing API key)."

    params = {
        "api_token": api_key,
        "search": query,
        "language": "en",
        "limit": MARKETAUX_MAX_ARTICLES,
    }

    try:
        response = requests.get(MARKETAUX_URL, params=params, timeout=MARKETAUX_TIMEOUT_SECONDS)
    except requests.exceptions.Timeout:
        return "Finance news service is currently unavailable (request timed out). Please try again later."
    except requests.exceptions.RequestException:
        return "Finance news service is currently unavailable (network error). Please try again later."

    if response.status_code != 200:
        return f"Finance news service returned an unexpected error (status {response.status_code})."

    try:
        data = response.json()
        articles = data.get("data", [])[:MARKETAUX_MAX_ARTICLES]

        if not articles:
            return f"No recent financial news found for '{query}'."

        lines = []
        for i, article in enumerate(articles, start=1):
            title = article.get("title", "Untitled")
            source = article.get("source", "Unknown source")
            published_at = article.get("published_at", "")
            description = article.get("description", "") or article.get("snippet", "")
            url = article.get("url", "")
            entities = article.get("entities", [])
            tickers = ", ".join(e.get("symbol", "") for e in entities if e.get("symbol")) or "N/A"

            lines.append(
                f"{i}. {title}\n"
                f"   Source: {source} | Published: {published_at} | Tickers: {tickers}\n"
                f"   {description}\n"
                f"   URL: {url}"
            )

        return "\n".join(lines)

    except (ValueError, TypeError):
        return "Received an unexpected response from the finance news service. Please try again later."

In [9]:
tools = [get_weather,web_search, finance_news]

print("Registered tools:")
for t in tools:
  print(f" - {t.name}: {t.description.strip().splitlines()[0]}")

Registered tools:
 - get_weather: Get current weather information for a specified location.
 - web_search: Search the web for current or general information.
 - finance_news: Search recent financial news using Marketaux.


In [10]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You are InfoPilot, a helpful assistant with access to three tools:

1. get_weather — current weather for a specific location.
2. web_search — general web search for current events and general knowledge.
3. finance_news — recent financial/stock-market news for a company or ticker.

Guidelines:
- Understand the user's intent before choosing a tool.
- Only use a tool when the answer requires current or external information;
  otherwise answer directly.
- Choose the tool whose purpose most specifically matches the request. A
  question that merely mentions a company name is NOT automatically a
  finance_news question — only use finance_news when the user is asking
  about financial/market news specifically (e.g. "latest news about X",
  "stock news for X", "earnings for X"). General "what is / who is / history
  of" questions about a company should use web_search instead.
- If a query needs more than one type of information (e.g. weather AND
  financial news), call multiple tools and combine their results into one
  answer.
- Never fabricate information that should come from a tool. If a tool
  reports an error or returns no results, say so honestly instead of
  guessing.
- Keep answers concise and clearly attribute information that came from a
  tool (e.g. mention the source name or link when the tool provided one).
- If a tool fails, briefly explain that the information could not be
  retrieved rather than making something up.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("Agent created.")


Agent created.


In [11]:
from langchain_core.messages import AIMessage, ToolMessage


def _extract_tool_calls(messages) -> list[str]:
    """
    Safely collect the names of every tool the agent called while answering.

    Reads AIMessage.tool_calls, which is the stable field across recent
    LangChain versions, rather than assuming one exact raw message shape.
    """
    tool_names = []
    for message in messages:
        if isinstance(message, AIMessage):
            for call in getattr(message, "tool_calls", None) or []:
                name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
                if name:
                    tool_names.append(name)
    return tool_names


def ask_agent(query: str, show_tools: bool = False) -> str:
    """
    Send a query to the InfoPilot agent and print its final answer.

    Args:
        query: The user's natural-language question.
        show_tools: If True, also print which tool(s) the agent used.

    Returns:
        The agent's final answer text.
    """
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    messages = result["messages"]

    final_message = messages[-1]
    final_text = getattr(final_message, "content", str(final_message))

    if show_tools:
        tools_used = _extract_tool_calls(messages)
        print(f"User Query:\n{query}\n")
        print(f"Tool(s) Used:\n{', '.join(tools_used) if tools_used else '(none — answered directly)'}\n")
        print(f"Final Answer:\n{final_text}")
    else:
        print(final_text)

    return final_text

testcase

In [12]:
ask_agent("what is the current weather of nagpur?",show_tools=True)

User Query:
what is the current weather of nagpur?

Tool(s) Used:
get_weather

Final Answer:
**Current weather in Nagpur, India**  
- Temperature: 23.6 °C (feels like 22.7 °C)  
- Humidity: 81 %  
- Wind: 24.1 kph  
- Precipitation: 0.02 mm (patchy rain nearby)  

*Source: get_weather tool*


'**Current weather in Nagpur, India**  \n- Temperature: 23.6\u202f°C (feels like 22.7\u202f°C)  \n- Humidity: 81\u202f%  \n- Wind: 24.1\u202fkph  \n- Precipitation: 0.02\u202fmm (patchy rain nearby)  \n\n*Source: get_weather tool*'

In [13]:
%pip install -q gradio

In [14]:
import gradio as gr
from langchain_core.messages import AIMessage

TOOL_EMOJIS = {"get_weather": "🌤️", "web_search": "🔎", "finance_news": "📈"}


def _extract_tool_names(messages) -> list[str]:
    """Collect the names of every tool the agent called while answering."""
    names = []
    for m in messages:
        if isinstance(m, AIMessage):
            for call in getattr(m, "tool_calls", None) or []:
                name = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
                if name:
                    names.append(name)
    return names


def user_submit(message, history):
    message = (message or "").strip()
    if not message:
        return gr.update(), history
    return "", history + [{"role": "user", "content": message}]


def bot_respond(history):
    """Call the notebook's existing `agent` with the chat so far and append its reply."""
    if not history or history[-1]["role"] != "user":
        return history

    context = [m for m in history[:-1] if m["role"] in ("user", "assistant")]
    messages = context + [{"role": "user", "content": history[-1]["content"]}]

    try:
        result = agent.invoke({"messages": messages})  # uses the `agent` already built above
        result_messages = result["messages"]
        answer = getattr(result_messages[-1], "content", "") or "(empty response)"
        tools_used = _extract_tool_names(result_messages)
    except Exception as exc:
        answer, tools_used = f"⚠️ Agent error: {exc}", []

    if tools_used:
        badge = "  ".join(f"{TOOL_EMOJIS.get(t, '🔧')} `{t}`" for t in tools_used)
        answer = f"*Tools used: {badge}*\n\n{answer}"

    return history + [{"role": "assistant", "content": answer}]


with gr.Blocks(title="InfoPilot (demo)") as demo:
    gr.Markdown("## 🛰️ InfoPilot — Multi-Tool AI Agent (notebook demo)")
    chatbot = gr.Chatbot(height=480, label="InfoPilot")
    msg = gr.Textbox(placeholder="Ask about weather, general facts, or financial news...", label="Your message")
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear")
    gr.Examples(
        examples=[
            "What is the current weather in Mumbai?",
            "Who is the current CEO of NVIDIA?",
            "What is the latest financial news about NVIDIA?",
            "What is the weather in Bengaluru and the latest financial news about Infosys?",
        ],
        inputs=msg,
    )

    msg.submit(user_submit, [msg, chatbot], [msg, chatbot]).then(bot_respond, chatbot, chatbot)
    send_btn.click(user_submit, [msg, chatbot], [msg, chatbot]).then(bot_respond, chatbot, chatbot)
    clear_btn.click(lambda: [], outputs=chatbot)

demo.launch(share=True, debug=False)  # share=True gives a public link — handy for Colab

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://770d74567a7327ab05.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
